In [1]:
# ============================================
# Step 5.1: Data Splitting (Addressing Overfitting)
# ============================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 1. Load the cleaned dataset
df = pd.read_csv('https://raw.githubusercontent.com/ThaLovelace/CS372_Mini-Project-1/refs/heads/main/Data/cleaned_sleep_data.csv', keep_default_na=False)

# 2. Use ONLY the 9 features selected from Chapter 4
selected_features = [
    'Systolic_BP', 'Sleep Duration', 'Daily Steps', 'BMI Category', 
    'Physical Activity Level', 'Diastolic_BP', 'Heart Rate', 'Age', 'Quality of Sleep'
]

X = df[selected_features]
y = df['Sleep Disorder']

# 3. Encode the Target labels (None=1, Insomnia=0, Sleep Apnea=2)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 4. Split data into 80% Training and 20% Testing
# Use 'stratify' to ensure the proportion of sleep disorders is the same in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

print("======================================")
print("Data Split Statistics")
print("======================================")
print(f"Total instances: {len(df)}")
print(f"Training set: {len(X_train)} samples (80%)")
print(f"Testing set: {len(X_test)} samples (20%)")
print("Target classes mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Data Split Statistics
Total instances: 374
Training set: 299 samples (80%)
Testing set: 75 samples (20%)
Target classes mapping: {'Insomnia': 0, 'None': 1, 'Sleep Apnea': 2}


In [2]:
# ============================================
# Step 5.2: k-Nearest Neighbors (kNN) with Grid Search
# ============================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Define the parameter grid for searching the best 'k' and distance metrics
# We test different numbers of neighbors (n_neighbors) and distance calculation methods
knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# 2. Initialize Grid Search with 5-fold Cross Validation
# This ensures that our hyperparameter tuning is robust and not biased
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5, scoring='accuracy')
knn_grid.fit(X_train, y_train)

# 3. Retrieve the best model found by Grid Search
best_knn = knn_grid.best_estimator_
knn_pred = best_knn.predict(X_test)

# 4. Display the results
print("--- kNN Optimization Results ---")
print(f"Best Parameters: {knn_grid.best_params_}")
print(f"Best Cross-Validation Accuracy: {knn_grid.best_score_:.4f}")
print(f"Final Test Accuracy: {accuracy_score(y_test, knn_pred):.4f}")

# 5. Detailed Classification Report
print("\nClassification Report (Test Set):")
print(classification_report(y_test, knn_pred, target_names=le.classes_))

--- kNN Optimization Results ---
Best Parameters: {'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'uniform'}
Best Cross-Validation Accuracy: 0.8795
Final Test Accuracy: 0.9467

Classification Report (Test Set):
              precision    recall  f1-score   support

    Insomnia       0.87      0.87      0.87        15
        None       1.00      0.98      0.99        44
 Sleep Apnea       0.88      0.94      0.91        16

    accuracy                           0.95        75
   macro avg       0.92      0.93      0.92        75
weighted avg       0.95      0.95      0.95        75



In [6]:
# ============================================
# Step 5.3: Decision Tree with Grid Search
# ============================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

# 1. Define the parameter grid for searching the best tree structure
# 'criterion' measures the quality of a split, 'max_depth' prevents the tree from growing too deep
dt_params = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 3, 5, 7, 10],
    'min_samples_split': [2, 5, 10]
}

# 2. Initialize Grid Search with 5-fold Cross Validation
# random_state=42 ensures the results are reproducible
dt_grid = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5, scoring='accuracy')
dt_grid.fit(X_train, y_train)

# 3. Retrieve the best model found by Grid Search
best_dt = dt_grid.best_estimator_
dt_pred = best_dt.predict(X_test)

# 4. Display the results
print("--- Decision Tree Optimization Results ---")
print(f"Best Parameters: {dt_grid.best_params_}")
print(f"Best Cross-Validation Accuracy: {dt_grid.best_score_:.4f}")
print(f"Final Test Accuracy: {accuracy_score(y_test, dt_pred):.4f}")

# 5. Detailed Classification Report
print("\nClassification Report (Test Set):")
print(classification_report(y_test, dt_pred, target_names=le.classes_))

--- Decision Tree Optimization Results ---
Best Parameters: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 5}
Best Cross-Validation Accuracy: 0.8863
Final Test Accuracy: 0.9733

Classification Report (Test Set):
              precision    recall  f1-score   support

    Insomnia       0.93      0.93      0.93        15
        None       1.00      1.00      1.00        44
 Sleep Apnea       0.94      0.94      0.94        16

    accuracy                           0.97        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.97      0.97      0.97        75



In [5]:
# ============================================
# Step 5.4: XGBoost Classifier with Grid Search 
# ============================================

import os
os.environ['OMP_NUM_THREADS'] = '1'

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
import joblib
import warnings

warnings.filterwarnings('ignore')

# 1. Define parameter grid for XGBoost
xgb_params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [50, 100]
}

# 2. Initialize XGBoost model with single thread settings
xgb_model = XGBClassifier(
    random_state=42, 
    n_jobs=1, 
    nthread=1,
    eval_metric='mlogloss'
)

# 3. Initialize Grid Search
xgb_grid = GridSearchCV(
    xgb_model, 
    xgb_params, 
    cv=5, 
    scoring='accuracy',
    n_jobs=1
)

print("Starting Grid Search... ")

# 4. Fit the model using joblib sequential backend to absolutely prevent deadlocks
with joblib.parallel_backend('sequential'):
    xgb_grid.fit(X_train, y_train)

# 5. Retrieve best model and predict
best_xgb = xgb_grid.best_estimator_
xgb_pred = best_xgb.predict(X_test)

# 6. Display results
print("--- XGBoost Optimization Results ---")
print(f"Best Parameters: {xgb_grid.best_params_}")
print(f"Best Cross-Validation Accuracy: {xgb_grid.best_score_:.4f}")
print(f"Final Test Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, xgb_pred))

Starting Grid Search... 
--- XGBoost Optimization Results ---
Best Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Best Cross-Validation Accuracy: 0.9030
Final Test Accuracy: 0.9333

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.81      0.87      0.84        15
           1       1.00      1.00      1.00        44
           2       0.87      0.81      0.84        16

    accuracy                           0.93        75
   macro avg       0.89      0.89      0.89        75
weighted avg       0.93      0.93      0.93        75



In [4]:
# ============================================
# Step 5.5: Neural Network (MLP) with Grid Search
# ============================================

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
import warnings

# Ignore convergence warnings to keep the output clean
warnings.filterwarnings('ignore') 

# 1. Define the parameter grid for the Neural Network
# 'hidden_layer_sizes' defines the architecture (neurons per layer)
# 'activation' defines the mathematical function used in the neurons
mlp_params = {
    'hidden_layer_sizes': [(50,), (100,), (50, 25)],
    'activation': ['relu', 'tanh'],
    'max_iter': [1500] # Set high enough to allow the network to converge
}

# 2. Initialize Grid Search with 5-fold Cross Validation
# n_jobs=1 is used to prevent deadlock/hanging on Windows systems
mlp_grid = GridSearchCV(MLPClassifier(random_state=42, solver='adam'), 
                        mlp_params, cv=5, scoring='accuracy', n_jobs=1)
mlp_grid.fit(X_train, y_train)

# 3. Retrieve the best model found by Grid Search
best_mlp = mlp_grid.best_estimator_
mlp_pred = best_mlp.predict(X_test)

# 4. Display the results
print("--- Neural Network Optimization Results ---")
print(f"Best Parameters: {mlp_grid.best_params_}")
print(f"Best Cross-Validation Accuracy: {mlp_grid.best_score_:.4f}")
print(f"Final Test Accuracy: {accuracy_score(y_test, mlp_pred):.4f}")

# 5. Detailed Classification Report
print("\nClassification Report (Test Set):")
print(classification_report(y_test, mlp_pred, target_names=le.classes_))

--- Neural Network Optimization Results ---
Best Parameters: {'activation': 'relu', 'hidden_layer_sizes': (100,), 'max_iter': 1500}
Best Cross-Validation Accuracy: 0.8963
Final Test Accuracy: 0.9600

Classification Report (Test Set):
              precision    recall  f1-score   support

    Insomnia       0.88      0.93      0.90        15
        None       1.00      0.98      0.99        44
 Sleep Apnea       0.94      0.94      0.94        16

    accuracy                           0.96        75
   macro avg       0.94      0.95      0.94        75
weighted avg       0.96      0.96      0.96        75

